# Energy Demand Forecast

A focused time-series notebook: create lag features from the past only, split in time order, and compare a model against naive baselines.

## Setup
This runs on the local energy-demand CSV bundled with the site. The sample is intentionally small, so focus on workflow discipline rather than score quality.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
DATA_PATH = '/cases/datasets/energy_demand_sample.csv'

def read_portal_csv(path):
    site_path = path if path.startswith('/') else f'/{path}'
    try:
        open_url = __import__('pyodide.http', fromlist=['open_url']).open_url
        return pd.read_csv(open_url(site_path))
    except Exception:
        relative = site_path.lstrip('/')
        candidates = [Path(relative), Path('..') / relative, Path.cwd() / relative, Path.cwd().parent / relative]
        for candidate in candidates:
            if candidate.exists():
                return pd.read_csv(candidate)
        raise FileNotFoundError(f'Could not find {site_path}')

raw = read_portal_csv(DATA_PATH)
raw['date'] = pd.to_datetime(raw['date'])
daily = raw.sort_values('date').set_index('date')
print('Rows:', len(daily))
display(daily)
display(daily[['demand_kwh', 'temp_c']].describe().T)

In [ ]:
series = daily.copy()
for lag in [1, 2, 3]:
    series[f'lag_{lag}'] = series['demand_kwh'].shift(lag)

series['rolling_3_mean'] = series['demand_kwh'].shift(1).rolling(3).mean()
series['rolling_3_std'] = series['demand_kwh'].shift(1).rolling(3).std().fillna(0)
series['day_index'] = np.arange(len(series))
series = series.dropna().copy()

features = [
    'temp_c',
    'is_weekend',
    'lag_1',
    'lag_2',
    'lag_3',
    'rolling_3_mean',
    'rolling_3_std',
    'day_index',
]
split = max(4, int(len(series) * 0.7))
train = series.iloc[:split].copy()
test = series.iloc[split:].copy()

model = RandomForestRegressor(
    n_estimators=160,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
)
model.fit(train[features], train['demand_kwh'])
test['model'] = model.predict(test[features])
test['naive_lag_1'] = test['lag_1']
test['rolling_baseline'] = test['rolling_3_mean']

score_rows = []
for name in ['model', 'naive_lag_1', 'rolling_baseline']:
    mae = mean_absolute_error(test['demand_kwh'], test[name])
    rmse = np.sqrt(mean_squared_error(test['demand_kwh'], test[name]))
    score_rows.append({'forecast': name, 'MAE': round(float(mae), 4), 'RMSE': round(float(rmse), 4)})

scores = pd.DataFrame(score_rows).sort_values('MAE')
display(scores)
display(test[['demand_kwh', 'model', 'naive_lag_1', 'rolling_baseline']])

plt.figure(figsize=(8, 4))
plt.plot(train.index, train['demand_kwh'], marker='o', label='train')
plt.plot(test.index, test['demand_kwh'], marker='o', label='actual')
plt.plot(test.index, test['model'], marker='o', label='model')
plt.plot(test.index, test['naive_lag_1'], marker='o', label='naive')
plt.title('Energy demand forecast')
plt.ylabel('kWh')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()